# PIPELINE 2: DATA ENRICHMENT & BUSINESS REPORTING
**Mục tiêu:**
1. Đọc lại Checkpoint (Dữ liệu 5K User & Keyword đã map tự động) từ Pipeline 1.
2. Tích hợp file Map tay của Data Engineer (Human-in-the-loop).
3. Map toàn bộ Category vào User Data.
4. Feature Engineering (Tạo label hành vi) và xuất Báo cáo chuyển dịch.

In [1]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F


spark = SparkSession.builder.appName("ETL_Pipeline_02_Enrich").getOrCreate()
CHECKPOINT_DIR = "/Users/hoaithuong/Desktop/Customer-Behavior-Analysis-Pipeline/data/checkpoints"

print("Đang phục hồi dữ liệu từ Checkpoint...")
df_5k = spark.read.parquet(f"{CHECKPOINT_DIR}/df_5k_checkpoint.parquet")
df_regex_success = spark.read.parquet(f"{CHECKPOINT_DIR}/df_regex_success.parquet")
# Định dạng lại để khớp cột
df_regex_success = df_regex_success.select(F.col("keyword").alias("Keyword_Gốc"), "category")

26/04/08 17:50:06 WARN Utils: Your hostname, Ryms.local resolves to a loopback address: 127.0.0.1; using 192.168.1.234 instead (on interface en0)
26/04/08 17:50:06 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/08 17:50:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Đang phục hồi dữ liệu từ Checkpoint...


In [2]:
# 2. TÍCH HỢP DỮ LIỆU TỪ HUMAN-IN-THE-LOOP
from asyncio.log import logger


print("Đang nạp file kết quả Map tay của Data Engineer...")
MANUAL_FILE = "/Users/hoaithuong/Desktop/Customer-Behavior-Analysis-Pipeline/data/mapping/Tu_Khoa_Da_Map_Tay.xlsx"

try:
    pdf_manual = pd.read_excel(MANUAL_FILE)
    # Ép gọt không nương tay để chống lỗi Join
    pdf_manual['Keyword_Gốc'] = pdf_manual['Keyword_Gốc'].astype(str).str.strip().str.lower()
    pdf_manual['Trưởng_Họ_(Family)'] = pdf_manual['Trưởng_Họ_(Family)'].astype(str).str.strip().str.lower()
    pdf_manual['category'] = pdf_manual['category'].astype(str).str.strip()
    
    df_manual_spark = spark.createDataFrame(pdf_manual[["Keyword_Gốc", "category"]])
    
    # Gộp thành Từ Điển Chuẩn (Master Dictionary)
    df_final_dictionary = df_regex_success.union(df_manual_spark)
    # Double check bằng Spark trim
    df_final_dictionary = df_final_dictionary.withColumn("Keyword_Gốc", F.trim(F.lower(F.col("Keyword_Gốc"))))
    
    print(f"Đã tạo thành công Master Dictionary với {df_final_dictionary.count()} từ khóa.")
except FileNotFoundError:
    print(f"LỖI: Không tìm thấy file '{MANUAL_FILE}'. Vui lòng hoàn thành Map tay ở Pipeline 1!")
    raise

Đang nạp file kết quả Map tay của Data Engineer...
Đã tạo thành công Master Dictionary với 5657 từ khóa.


In [3]:
# 3. BULLETPROOF JOIN VÀO MASTER DATA
print("Đang ánh xạ (Mapping) Thể loại vào 5.000 User...")

# Đảm bảo Data gốc không chứa khoảng trắng tàng hình
df_5k_safe = df_5k.withColumn("most_search_t6", F.trim(F.lower(F.col("most_search_t6")))) \
                  .withColumn("most_search_t7", F.trim(F.lower(F.col("most_search_t7"))))

# Map cho tháng 6
df_final = df_5k_safe.join(
    F.broadcast(df_final_dictionary).select(F.col("Keyword_Gốc").alias("kw_t6"), F.col("category").alias("category_t6")),
    df_5k_safe.most_search_t6 == F.col("kw_t6"),
    how="left"
).drop("kw_t6")

# Map cho tháng 7
df_final = df_final.join(
    F.broadcast(df_final_dictionary).select(F.col("Keyword_Gốc").alias("kw_t7"), F.col("category").alias("category_t7")),
    df_final.most_search_t7 == F.col("kw_t7"),
    how="left"
).drop("kw_t7")

# Xử lý các rớt mạng và gán nhãn No_Search chuẩn nghiệp vụ
df_final = df_final.fillna("Other", subset=["category_t6", "category_t7"])
df_final = df_final.withColumn(
    "category_t6", F.when(F.col("most_search_t6").isNull(), "No_Search").otherwise(F.col("category_t6"))
).withColumn(
    "category_t7", F.when(F.col("most_search_t7").isNull(), "No_Search").otherwise(F.col("category_t7"))
)

Đang ánh xạ (Mapping) Thể loại vào 5.000 User...


In [6]:
df_final_enriched = df_final.withColumn(
    "user_status",
    F.when(F.col("category_t6") == "No_Search", "New_User")       
     .when(F.col("category_t7") == "No_Search", "Churned_User")   
     .otherwise("Retained_User")                                  
).withColumn(
    "trending_types",
    F.when(F.col("category_t6") == F.col("category_t7"), "unchanged").otherwise("changed")
).withColumn(
    "Previous",
    F.when(
        F.col("trending_types") == "unchanged", F.col("category_t6") 
    ).otherwise(
        F.concat(F.col("category_t6"), F.lit(" -> "), F.col("category_t7")) 
    )
).cache() # Khóa dữ liệu cuối cùng vào RAM để báo cáo nhanh hơn

df_final_enriched = df_final_enriched.select("user_id","most_search_t6", "category_t6", "most_search_t7", "category_t7", "user_status", "trending_types", "Previous")
df_final_enriched.show(10, truncate=False)

+--------+-------------------------------------------------+--------------+------------------------------------------------------+--------------+-------------+--------------+-------------------------+
|user_id |most_search_t6                                   |category_t6   |most_search_t7                                        |category_t7   |user_status  |trending_types|Previous                 |
+--------+-------------------------------------------------+--------------+------------------------------------------------------+--------------+-------------+--------------+-------------------------+
|49081622|kung fu masters of the zodiac: 12 zodiac way     |Cartoon_Kids  |johnny english                                        |Education     |Retained_User|changed       |Cartoon_Kids -> Education|
|5839440 |xem phim việt nam                                |Movies_General|tập dưỡng sinh trung quốc                             |K-Drama       |Retained_User|changed       |Movies_General -> K-Dr

In [7]:
# 5. BUSINESS REPORTING & EXPORT
print("Trích xuất Ma trận Chuyển dịch Hành vi (Transition Matrix)...")

df_transition_matrix = df_final_enriched.groupBy("category_t6").pivot("category_t7").count().fillna(0)
df_transition_matrix.orderBy("category_t6").show(30, truncate=False)

# Xuất file Final
df_transition_matrix.toPandas().to_excel("Bao_Cao_Chuyen_Dich_Thang_6_7.xlsx", index=False)
df_final_enriched.write.mode("overwrite").parquet("Master_Table_Enriched.parquet")

print("🎉 TẤT CẢ PIPELINE ĐÃ HOÀN TẤT THÀNH CÔNG! Báo cáo đã được lưu.")

Trích xuất Ma trận Chuyển dịch Hành vi (Transition Matrix)...
+--------------+------+-----+-----+-------+------------+---------+------+-------+--------------+-----+----+---------+------+------+-------+----------+-------+-------+
|category_t6   |Action|Adult|Anime|C-Drama|Cartoon_Kids|Education|Horror|K-Drama|Movies_General|Music|News|No_Search|Sports|System|TV_Show|Thai-Drama|Unknown|V-Drama|
+--------------+------+-----+-----+-------+------------+---------+------+-------+--------------+-----+----+---------+------+------+-------+----------+-------+-------+
|Action        |17    |1    |10   |7      |5           |1        |9     |15     |32            |7    |0   |0        |2     |5     |1      |1         |4      |2      |
|Adult         |0     |3    |2    |0      |0           |2        |2     |2      |6             |0    |1   |0        |0     |1     |0      |1         |1      |0      |
|Anime         |14    |3    |306  |25     |24          |8        |58    |34     |69            |11   |0

26/04/08 20:22:31 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 977597 ms exceeds timeout 120000 ms
26/04/08 20:22:31 WARN SparkContext: Killing executors is not supported by current scheduler.
26/04/08 20:22:33 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$